# ANN for fashion mnist pytorch GPU Optimized

to reduce overfitting, we are using `Dropouts`
<br>
**Dropout**
- Applied to hidden layers
- Applied after the ReLU activation functions.
- Randoml turns off p% neurons in hidden layer during each forward pass.
- Has a regularization effect
- During evaluaion dropout is used.

In [13]:
import kagglehub
path = kagglehub.dataset_download("zalando-research/fashionmnist")
print(path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
/kaggle/input/fashionmnist


In [14]:
import os

for root,dirs,files in os.walk(path):
  for file in files:
    print(os.path.join(root,file))

/kaggle/input/fashionmnist/t10k-labels-idx1-ubyte
/kaggle/input/fashionmnist/t10k-images-idx3-ubyte
/kaggle/input/fashionmnist/fashion-mnist_test.csv
/kaggle/input/fashionmnist/fashion-mnist_train.csv
/kaggle/input/fashionmnist/train-labels-idx1-ubyte
/kaggle/input/fashionmnist/train-images-idx3-ubyte


In [15]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [16]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using {device}")

using cuda


In [17]:
train_df = pd.read_csv(os.path.join(path, "fashion-mnist_train.csv"))
test_df=pd.read_csv(os.path.join(path,"fashion-mnist_test.csv"))
X_train=train_df.drop(columns=["label"]).values
y_train=train_df["label"].values

X_test=test_df.drop(columns=["label"]).values
y_test=test_df["label"].values

X_train,X_val,y_train,y_val=train_test_split(X_train,y_train,test_size=0.2,random_state=42,stratify=y_train)

In [18]:
# Normalize
X_train = X_train.astype(np.float32) / 255.0
X_val = X_val.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [19]:
from torch.utils.data import TensorDataset
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)



In [20]:
class MyNN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):

    super().__init__()

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self, x):

    return self.model(x)

In [21]:
# objective function
def objective(trial):

  # next hyperparameter values from the search space
  num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
  epochs = trial.suggest_int("epochs", 10, 50, step=10)
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
  dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
  val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

  # model init
  input_dim = 784
  output_dim = 10

  model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model.to(device)

  # optimizer selection
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)

  if optimizer_name == 'Adam':
    optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop

  for epoch in range(epochs):

    for batch_features, batch_labels in train_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # calculate loss
      loss = criterion(outputs, batch_labels)

      # back pass
      optimizer.zero_grad()
      loss.backward()

      # update grads
      optimizer.step()


  # evaluation
  model.eval()
  # evaluation on test data
  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in val_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct/total

  return accuracy

In [22]:
!pip install optuna

In [23]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)
study.best_value
study.best_params

[I 2026-06-08 04:41:35,395] A new study created in memory with name: no-name-523ac1e0-884e-439f-aca0-63651e90cfef
[I 2026-06-08 04:42:22,380] Trial 0 finished with value: 0.8434166666666667 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 24, 'epochs': 20, 'learning_rate': 4.595018441485546e-05, 'dropout_rate': 0.30000000000000004, 'batch_size': 64, 'optimizer': 'RMSprop', 'weight_decay': 3.167046961136937e-05}. Best is trial 0 with value: 0.8434166666666667.
[I 2026-06-08 04:44:21,045] Trial 1 finished with value: 0.8766666666666667 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 48, 'epochs': 50, 'learning_rate': 0.0038024631224856794, 'dropout_rate': 0.2, 'batch_size': 64, 'optimizer': 'RMSprop', 'weight_decay': 0.000528705969180005}. Best is trial 1 with value: 0.8766666666666667.
[I 2026-06-08 04:45:20,687] Trial 2 finished with value: 0.87375 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 40, 'epochs': 50, 'learning_rate': 0.0005628805567

{'num_hidden_layers': 2,
 'neurons_per_layer': 72,
 'epochs': 10,
 'learning_rate': 1.3940721047780901e-05,
 'dropout_rate': 0.2,
 'batch_size': 32,
 'optimizer': 'RMSprop',
 'weight_decay': 0.00013133399581960437}